#### Import Libraries

In [73]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
print("All good 🚀")

All good 🚀


##### Load the dataset

In [213]:
model_df = pd.read_csv("../data/processed/bed_occupancy_modelling_dataset.csv")

#### Create more features

In [214]:
model_df["over_capacity"] = (
    model_df["occupancy_rate"] > 1
).astype(int)

In [215]:
model_df["available_beds"] = (
    model_df["available_beds"]
    .clip(lower=0)
)

#### Sort Date Column

In [216]:
model_df["date"] = pd.to_datetime(
    model_df["date"]
)

In [217]:
model_df = model_df.sort_values(
    [
        "hospital_id",
        "ward",
        "date"
    ]
)

In [218]:
model_df.duplicated(
    [
        "hospital_id",
        "ward",
        "date"
    ]
).sum()

0

##### Check Date Coverage - continuous dates for forecasting

In [219]:
model_df.groupby(
    [
        "hospital_id",
        "ward"
    ]
)["date"].agg(
    ["min","max","count"]
)

min        max  count
hospital_id ward                                                
HHN-BIR-01  Cardiology Ward         2024-01-01 2025-12-31    731
            Day Case Unit           2024-01-01 2025-12-31    731
            General Medicine Ward A 2024-01-01 2025-12-31    731
            General Medicine Ward B 2024-01-01 2025-12-31    731
            Icu                     2024-01-01 2025-12-31    731
            Oncology Ward           2024-01-01 2025-12-31    731
            Orthopaedics Ward A     2024-01-01 2025-12-31    731
            Orthopaedics Ward B     2024-01-01 2025-12-31    731
HHN-EDI-01  Cardiology Ward         2024-01-01 2025-12-31    731
            Day Case Unit           2024-01-01 2025-12-31    731
            General Medicine Ward A 2024-01-01 2025-12-31    731
            General Medicine Ward B 2024-01-01 2025-12-31    731
            Icu                     2024-01-01 2025-12-31    731
            Oncology Ward           2024-01-01 2025-12-31    731
            Orthopaedics Ward A     2024-01-01 2025-12-31    731
            Orthopaedics Ward B     2024-01-01 2025-12-31    731
HHN-LON-01  Cardiology Ward         2024-01-01 2025-12-31    731
            Day Case Unit           2024-01-01 2025-12-31    731
            General Medicine Ward A 2024-01-01 2025-12-31    731
            General Medicine Ward B 2024-01-01 2025-12-31    731
            Icu                     2024-01-01 2025-12-31    731
            Oncology Ward           2024-01-01 2025-12-31    731
            Orthopaedics Ward A     2024-01-01 2025-12-31    731
            Orthopaedics Ward B     2024-01-01 2025-12-31    731
HHN-LON-02  Cardiology Ward         2024-01-01 2025-12-31    731
            Day Case Unit           2024-01-01 2025-12-31    731
            General Medicine Ward A 2024-01-01 2025-12-31    731
            General Medicine Ward B 2024-01-01 2025-12-31    731
            Icu                     2024-01-01 2025-12-31    731
            Oncology Ward           2024-01-01 2025-12-31    731
            Orthopaedics Ward A     2024-01-01 2025-12-31    731
            Orthopaedics Ward B     2024-01-01 2025-12-31    731
HHN-MAN-01  Cardiology Ward         2024-01-01 2025-12-31    731
            Day Case Unit           2024-01-01 2025-12-31    731
            General Medicine Ward A 2024-01-01 2025-12-31    731
            General Medicine Ward B 2024-01-01 2025-12-31    731
            Icu                     2024-01-01 2025-12-31    731
            Oncology Ward           2024-01-01 2025-12-31    731
            Orthopaedics Ward A     2024-01-01 2025-12-31    731
            Orthopaedics Ward B     2024-01-01 2025-12-31    731

##### Add Time-Series Features

##### Lag 1 - Yesterday's occupancy

In [220]:
model_df["lag_1_occupied_beds"] = (
    model_df
    .groupby(
        [
            "hospital_id",
            "ward"
        ]
    )
    ["occupied_beds"]
    .shift(1)
)

##### Lag 7 - One week ago (Previous 7 days)

In [221]:
model_df["lag_7_occupied_beds"] = (
    model_df
    .groupby(
        [
            "hospital_id",
            "ward"
        ]
    )
    ["occupied_beds"]
    .shift(7)
)

##### 7-day moving average

In [222]:
model_df["rolling_7_day_avg"] = (
    model_df
    .groupby(
        [
            "hospital_id",
            "ward"
        ]
    )
    ["occupied_beds"]
    .transform(
        lambda x:
        x.rolling(7).mean()
    )
)

##### Rolling 7-day Average

In [223]:
model_df["rolling_7_day_avg"] = (
    model_df.groupby(["hospital_id", "ward"])["occupied_beds"]
    .transform(lambda x: x.shift(1).rolling(7).mean())
)

##### Rolling 7-day Standard deviation

In [224]:
model_df["rolling_7_day_std"] = (
    model_df.groupby(["hospital_id", "ward"])["occupied_beds"]
    .transform(lambda x: x.shift(1).rolling(7).std())
)

##### Day of week

In [225]:
model_df["day_of_week"] = (
    model_df["date"]
    .dt.dayofweek
)

##### Month

In [226]:
model_df["month"] = (
    model_df["date"].dt.month
)

##### Length of Stay by day

In [227]:
model_df["avg_los_days"] = (
    model_df["avg_los_hours"] / 24
).round(1)

##### Day-to-Day Change

In [228]:
model_df["occupied_beds_change"] = (
    model_df.groupby(
        [
            "hospital_id",
            "ward"
        ]
    )["occupied_beds"]
    .diff()
)

##### Weekend

In [229]:
model_df["is_weekend"] = (
    model_df["date"].dt.dayofweek >= 5
).astype(int)

In [230]:
import holidays
uk_holidays = holidays.UnitedKingdom()

In [231]:
model_df["is_holiday"] = (
    model_df["date"]
    .isin(uk_holidays)
    .astype(int)
)

##### Fill Lag NA Values

In [232]:
lag_columns = [
    "lag_1_occupied_beds",
    "lag_7_occupied_beds",
    "rolling_7_day_avg",
    "rolling_7_day_std",
    "occupied_beds_change"
]

model_df[lag_columns] = (
    model_df[lag_columns]
    .fillna(0)
)

In [233]:
print("Shape:", model_df.shape)

model_df.info()

Shape: (29240, 35)
<class 'pandas.DataFrame'>
RangeIndex: 29240 entries, 0 to 29239
Data columns (total 35 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   hospital_id           29240 non-null  str           
 1   ward                  29240 non-null  str           
 2   bed_type              29240 non-null  str           
 3   date                  29240 non-null  datetime64[us]
 4   total_beds            29240 non-null  int64         
 5   staffed_beds          29240 non-null  int64         
 6   occupied_beds         29240 non-null  int64         
 7   closed_beds           29240 non-null  int64         
 8   occupancy_rate        29240 non-null  float64       
 9   available_beds        29240 non-null  int64         
 10  daily_admissions      29240 non-null  float64       
 11  avg_los_hours         29240 non-null  float64       
 12  emergency_admissions  29240 non-null  float64       
 13  planned_

In [234]:
missing = (
    model_df.isnull()
    .sum()
    .sort_values(ascending=False)
)

print(missing)

hospital_id             0
lag_7_occupied_beds     0
tier                    0
specialty_emphasis      0
total_bed_capacity      0
daily_discharges        0
over_capacity           0
lag_1_occupied_beds     0
rolling_7_day_avg       0
hospital_name           0
rolling_7_day_std       0
day_of_week             0
month                   0
avg_los_days            0
occupied_beds_change    0
is_weekend              0
city                    0
scheduled_surgeries     0
ward                    0
occupancy_rate          0
bed_type                0
date                    0
total_beds              0
staffed_beds            0
occupied_beds           0
closed_beds             0
available_beds          0
daily_ed_arrivals       0
daily_admissions        0
avg_los_hours           0
emergency_admissions    0
planned_staff           0
actual_staff            0
staffing_ratio          0
is_holiday              0
dtype: int64


In [235]:
duplicates = model_df.duplicated(
    subset=[
        "hospital_id",
        "ward",
        "bed_type",
        "date"
    ]
).sum()

print("Duplicate records:", duplicates)

Duplicate records: 0


In [236]:
print(
    model_df["occupied_beds"].describe()
)

count    29240.000000
mean        14.114603
std          8.732209
min          0.000000
25%          9.000000
50%         13.000000
75%         19.000000
max         43.000000
Name: occupied_beds, dtype: float64


#### Feature Engineering

##### Calendar Features

In [237]:
model_df["date"] = pd.to_datetime(model_df["date"])

model_df["year"] = model_df["date"].dt.year

model_df["month"] = model_df["date"].dt.month

model_df["day"] = model_df["date"].dt.day

model_df["day_of_week"] = model_df["date"].dt.dayofweek

model_df["week_of_year"] = model_df["date"].dt.isocalendar().week.astype(int)

model_df["quarter"] = model_df["date"].dt.quarter

model_df["is_weekend"] = (
    model_df["day_of_week"] >= 5
).astype(int)

##### Admission Pressure

In [238]:
model_df["admission_pressure"] = (
    model_df["daily_admissions"] /
    model_df["staffed_beds"]
).round(2)

##### Bed Pressure

In [239]:
model_df["bed_pressure"] = (
    model_df["occupied_beds"]
    -
    model_df["staffed_beds"]
)

##### Available Bed Percentage

In [240]:
model_df["available_bed_pct"] = (
    model_df["available_beds"]
    /
    model_df["staffed_beds"]
).round(1)

##### Staff Shortage

In [241]:
model_df["staff_shortage"] = (
    model_df["planned_staff"]
    -
    model_df["actual_staff"]
)

##### Emergency Admission Ratio

In [242]:
model_df["emergency_ratio"] = (
    model_df["emergency_admissions"]
    /
    model_df["daily_admissions"]
).round(1)

In [243]:
model_df["emergency_ratio"] = (
    model_df["emergency_ratio"]
    .fillna(0)
)

##### Surgery Pressure

In [244]:
model_df["surgery_per_staff"] = (
    model_df["scheduled_surgeries"]
    /
    model_df["actual_staff"]
).round(1)

model_df["surgery_per_staff"] = (
    model_df["surgery_per_staff"]
    .fillna(0)
)

In [245]:
model_df.head()

,hospital_id,ward,bed_type,date,total_beds,staffed_beds,occupied_beds,closed_beds,occupancy_rate,available_beds,daily_admissions,avg_los_hours,emergency_admissions,planned_staff,actual_staff,staffing_ratio,daily_ed_arrivals,scheduled_surgeries,hospital_name,city,tier,specialty_emphasis,total_bed_capacity,daily_discharges,over_capacity,lag_1_occupied_beds,lag_7_occupied_beds,rolling_7_day_avg,rolling_7_day_std,day_of_week,month,avg_los_days,occupied_beds_change,is_weekend,is_holiday,year,day,week_of_year,quarter,admission_pressure,bed_pressure,available_bed_pct,staff_shortage,emergency_ratio,surgery_per_staff
0,HHN-BIR-01,Cardiology Ward,Standard,2024-01-01,42,42,3,0,0.1,39,6.0,84.0,6.0,20,20,1.0,80,0.0,Horizon Birmingham,Birmingham,Medium,Cardiology,126,0.0,0,0.0,0.0,0.0,0.0,0,1,3.5,0.0,0,0,2024,1,1,1,0.14,-39,0.9,0,1.0,0.0
1,HHN-BIR-01,Cardiology Ward,Standard,2024-01-02,42,41,10,0,0.2,31,10.0,102.0,9.0,20,17,0.8,83,3.0,Horizon Birmingham,Birmingham,Medium,Cardiology,126,0.0,0,3.0,0.0,0.0,0.0,1,1,4.2,7.0,0,0,2024,2,1,1,0.24,-31,0.8,3,0.9,0.2
2,HHN-BIR-01,Cardiology Ward,Standard,2024-01-03,42,42,18,0,0.4,24,7.0,70.0,6.0,20,20,1.0,72,2.0,Horizon Birmingham,Birmingham,Medium,Cardiology,126,5.0,0,10.0,0.0,0.0,0.0,2,1,2.9,8.0,0,0,2024,3,1,1,0.17,-24,0.6,0,0.9,0.1
3,HHN-BIR-01,Cardiology Ward,Standard,2024-01-04,42,41,19,0,0.5,22,7.0,66.0,5.0,20,20,1.0,95,9.0,Horizon Birmingham,Birmingham,Medium,Cardiology,126,4.0,0,18.0,0.0,0.0,0.0,3,1,2.8,1.0,0,0,2024,4,1,1,0.17,-22,0.5,0,0.7,0.4
4,HHN-BIR-01,Cardiology Ward,Standard,2024-01-05,42,42,22,0,0.5,20,6.0,94.0,2.0,20,18,0.9,80,14.0,Horizon Birmingham,Birmingham,Medium,Cardiology,126,5.0,0,19.0,0.0,0.0,0.0,4,1,3.9,3.0,0,0,2024,5,1,1,0.14,-20,0.5,2,0.3,0.8


##### Save time series dataset

In [246]:
model_df.to_csv(
    "../data/processed/bed_occupancy_timeseries.csv",
    index=False
)